# Build BUSCO Gene Based Phylogenetic Trees for Cryptococcus and Relatives

In [ ]:
## Import dependencies

import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tarfile

In [ ]:
## Define primary taxa of interest (the taxa specified in folders/filenames)
#Taxa = "Cryptococcus"

## Load BUSCOs and Remove Low BUSCO Percentage Assemblies

#### First, load list of Cryptococcus accessions and identify this subset of BUSCO results zip files

In [ ]:
## Load list of fungi assemblies in GenBank, then remove duplicate accessions due to RefSeq
## Also add genus and species columns for filtering

# Change to appropriate directory containing summary of assemblies downloaded from GenBank
os.chdir('/path/to/your/analysis_directory/AccessionsForLINs/NCBIDownload_WeekOfOct23_2024')

# Replace 'file.tsv' with your actual file path
fungi_df = pd.read_csv('AllNCBI-FungalGenomes_Oct23_2024.tsv', sep='\t')

# Create a mask for GCF rows
gcf_mask = fungi_df['Assembly Accession'].str.startswith('GCF')

# Create a DataFrame of GCFs
gcf_df = fungi_df[gcf_mask]

# Create a new column in the original DataFrame to indicate presence of GCFs
# Initialize the column with False
fungi_df['Has GCF'] = False

# Set Has GCF to True for the corresponding GCAs
# Assuming GCA accessions correspond directly to GCF accessions by removing the 'GCF_' prefix
for gcf_accession in gcf_df['Assembly Accession']:
    gca_accession = gcf_accession.replace('GCF', 'GCA')  # Modify according to your naming convention
    if gca_accession in fungi_df['Assembly Accession'].values:
        fungi_df.loc[fungi_df['Assembly Accession'] == gca_accession, 'Has GCF'] = True

# Filter out rows where 'Assembly Accession' starts with 'GCF'
fungi_df = fungi_df[~gcf_mask]

# Reset index after deleting rows
fungi_df = fungi_df.reset_index(drop=True)

# Filter out rows where 'Assembly Accession' starts with 'GCF'
fungi_df = fungi_df[~fungi_df['Assembly Accession'].str.startswith('GCF')]

# Reset index after deleting rows
fungi_df = fungi_df.reset_index(drop=True)

# Add the genus column (extracting from Organism Name)
fungi_df['Genus'] = fungi_df['Organism Name'].apply(lambda x: x.split()[0])  # Assuming the genus is the first word
fungi_df['Species'] = fungi_df['Organism Name'].apply(lambda x: x.split()[1])  # Assuming the species is the first word


In [ ]:
## See how many assemblies there are for a particular genus and related genera (from looking at publications)
## If not enough relatives have assemblies, will need to go to higher taxa

Cryptococcus = ["Cryptococcus", "Tremella"]

# Create a new DataFrame 
Cryptococcus_df = fungi_df[fungi_df['Genus'].isin(Cryptococcus)]

# Count the occurrences of each unique genus and create a DataFrame
Genus_counts = Cryptococcus_df['Genus'].value_counts().reset_index()
Genus_counts.columns = ['Genus', 'Count']  # Rename columns

# Create a horizontal bar chart
plt.figure(figsize=(10, 9))
bar_plot = sns.barplot(x='Count', y='Genus', data=Genus_counts)

# Annotate the bars with counts
for p in bar_plot.patches:
    bar_plot.annotate(f'{int(p.get_width())}',  # Text to display
                      (p.get_width() + 0.5, p.get_y() + 0.2 + p.get_height()/2),  # Position
                       fontsize=15, color='black')

plt.xlabel('Count')
plt.ylabel('Genus')
plt.title('Number of Assemblies in Genbank')
plt.show()

In [ ]:
## See how many assemblies there are for a particular genus and related genera (from looking at publications)
## If not enough relatives have assemblies, will need to go to higher taxa

Cryptococcus = ["Cryptococcus", "Tremella"]

# Create a new DataFrame 
Cryptococcus_df = fungi_df[fungi_df['Genus'].isin(Cryptococcus)]

# Count the occurrences of each unique organism name and create a DataFrame
Species_counts = Cryptococcus_df['Organism Name'].value_counts().reset_index()
Species_counts.columns = ['Organism Name', 'Count']  # Rename columns

# Create a horizontal bar chart
plt.figure(figsize=(10, 14))
bar_plot = sns.barplot(x='Count', y='Organism Name', data=Species_counts)

# Annotate the bars with counts
for p in bar_plot.patches:
    bar_plot.annotate(f'{int(p.get_width())}',  # Text to display
                      (p.get_width() + 0.5, p.get_y() + 0.2 + p.get_height()/2),  # Position
                       fontsize=15, color='black')

plt.xlabel('Count')
plt.ylabel('Organism Name')
plt.title('Number of Assemblies in Genbank')
plt.show()

In [ ]:
## Make list of the zip folders that correspond to selected taxa BUSCO results

# Switch to directory containing zip files of BUSCO results
os.chdir("/path/to/your/analysis_directory/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/BUSCO_CryptococcusReduced/busco_results")

# Extract accessions from zip file folder names for cross-referencing with list of selected Cryptococcus accessions
AllfungiBUSCOs_ZipFiles = os.listdir(os.getcwd())
AllfungiBUSCOs_ZipAccessions = ["_".join(x.split("_")[3:]) for x in AllfungiBUSCOs_ZipFiles]
AllfungiBUSCOs_ZipAccessions = [".".join(x.split(".")[:2]) for x in AllfungiBUSCOs_ZipAccessions]

# Make list of Cryptococcus accessions from dataframe containing Cryptococcus GenBank information
CryptococcusList = list(Cryptococcus_df["Assembly Accession"])

# Find indices of elements in list of fungi zip files that are also in list of selected taxa
SelectedBUSCOs_ZipAccessionIndices = [i for i, x in enumerate(AllfungiBUSCOs_ZipAccessions) if x in CryptococcusList]

# Use indices determined above to extract filenames of selected taxa BUSCO results as new list
SelectedBUSCOs_ZipFiles = [AllfungiBUSCOs_ZipFiles[i] for i in SelectedBUSCOs_ZipAccessionIndices]

In [ ]:
## Sanity check that the correct number of zip files are listed

print(len(CryptococcusList))
print(len(SelectedBUSCOs_ZipFiles))

## Now reduce the number of assemblies as there are too many isolates to focus on the species complexes in the trees produced with this data

## The plan is to remove all "Cryptococcus neoformans" with no further description. Remove 50 of 61 of isolates names "Cryptococcus neoformans serotype A". Remove all Tremella besides "Tremella fuciformis" and "Tremella yokohamiensis". In the prior step, all Kwoniella and Bullera were removed. This should reduce from like 254 or so to under 75

In [ ]:
Cryptococcus_df

In [ ]:
# Create a copy of the original dataframe to work with
filtered_df = Cryptococcus_df.copy()

# Track what's being removed
removed_counts = {}

# 1. Remove all Tremella that isn't "Tremella fuciformis" or "Tremella yokohamaensis"
tremella_to_remove = filtered_df['Genus'].eq('Tremella') & ~filtered_df['Organism Name'].str.contains('Tremella fuciformis', case=False, na=False)
removed_counts['Tremella (non-fuciformis)'] = tremella_to_remove.sum()
filtered_df = filtered_df[~tremella_to_remove]

# 2. Remove all Cryptococcus that includes "sp." (unnamed species)
cryptococcus_sp_to_remove = filtered_df['Genus'].eq('Cryptococcus') & filtered_df['Organism Name'].str.contains('sp\.', case=False, na=False)
removed_counts['Cryptococcus sp.'] = cryptococcus_sp_to_remove.sum()
filtered_df = filtered_df[~cryptococcus_sp_to_remove]

# 3. Remove any that are just "Cryptococcus neoformans" (no additional descriptors)
just_neoformans_to_remove = filtered_df['Organism Name'].eq('Cryptococcus neoformans')
removed_counts['Just "Cryptococcus neoformans"'] = just_neoformans_to_remove.sum()
filtered_df = filtered_df[~just_neoformans_to_remove]

# 4. Remove 61 random isolates that are "Cryptococcus neoformans serotype A" (that's all of them... thought I would do less, but ended up removing all)
serotype_a_isolates = filtered_df[filtered_df['Organism Name'].str.contains('Cryptococcus neoformans.*serotype A', case=False, na=False)]
removed_counts['Cryptococcus neoformans serotype A'] = min(61, len(serotype_a_isolates))

if len(serotype_a_isolates) > 61:
    indices_to_remove = serotype_a_isolates.sample(n=61, random_state=42).index
    filtered_df = filtered_df.drop(indices_to_remove)
else:
    filtered_df = filtered_df.drop(serotype_a_isolates.index)

In [ ]:
## See how many assemblies there are for a particular genus and related genera (from looking at publications)
## If not enough relatives have assemblies, will need to go to higher taxa


# Count the occurrences of each unique organism name and create a DataFrame
Species_counts = filtered_df['Organism Name'].value_counts().reset_index()
Species_counts.columns = ['Organism Name', 'Count']  # Rename columns

# Create a horizontal bar chart
plt.figure(figsize=(10, 14))
bar_plot = sns.barplot(x='Count', y='Organism Name', data=Species_counts)

# Annotate the bars with counts
for p in bar_plot.patches:
    bar_plot.annotate(f'{int(p.get_width())}',  # Text to display
                      (p.get_width() + 0.5, p.get_y() + 0.2 + p.get_height()/2),  # Position
                       fontsize=15, color='black')

plt.xlabel('Count')
plt.ylabel('Organism Name')
plt.title('Number of Assemblies in Genbank')
plt.show()

In [ ]:
Cryptococcus_df = filtered_df

Cryptococcus_df

In [ ]:
# Make list of Cryptococcus accessions from dataframe containing Cryptococcus GenBank information
CryptococcusList = list(Cryptococcus_df["Assembly Accession"])

# Find indices of elements in list of fungi zip files that are also in list of selected taxa
SelectedBUSCOs_ZipAccessionIndices = [i for i, x in enumerate(AllfungiBUSCOs_ZipAccessions) if x in CryptococcusList]

# Use indices determined above to extract filenames of selected taxa BUSCO results as new list
SelectedBUSCOs_ZipFiles = [AllfungiBUSCOs_ZipFiles[i] for i in SelectedBUSCOs_ZipAccessionIndices]

In [ ]:
len(SelectedBUSCOs_ZipFiles)

In [ ]:
SelectedBUSCOs_ZipFiles

In [ ]:
## Determine the completeness and single copy number directly from BUSCO results compressed tarfiles

# Path to the tar.gz file
file_path = '/path/to/your/analysis_directory/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/BUSCO_CryptococcusReduced/busco_results/'

# Create lists for containing results of computations
Completeness = []
SingleCopies = []
Duplicates = []
Fragments = []
Missing = []
count = 0     # Initialize counter used in loop

for i in np.arange(len(SelectedBUSCOs_ZipFiles)):
    count += 1     # Count number of files operated on
    
    # Tarfile operations
    with tarfile.open(file_path+SelectedBUSCOs_ZipFiles[i], 'r:gz') as tar:
    
        # Read the contents of the tar file
        fileList = tar.getnames()      # get filenames for content of tar file
        print("Running for file: ", fileList[0], "     File#"+str(count))     # Print for intermittent status update
        # BUSCO names this file short_summary.specific.<lineage>.<output_name>.txt, not
        # a literal 'short_summary.txt', so find it by prefix/suffix instead of a fixed name
        summary_name = next(f for f in fileList if os.path.basename(f).startswith('short_summary') and f.endswith('.txt'))
        member = tar.getmember(summary_name)     # Select file of interest
        file_content = tar.extractfile(member).read()    # Read file
        lines = file_content.decode('utf-8').split('\n')     # Decode from bytes to readable file, and delimit new lines

        # Compute completeness and number of single copy BUSCO genes 
        #(the following delimits by tabs and selects the appropriate numbers from the BUSCO summary text file)
        NumberComplete = int(lines[9].split('\t')[1])
        NumberSingleCopy = int(lines[10].split('\t')[1])
        NumberDuplicate = int(lines[11].split('\t')[1])
        NumberFragment = int(lines[12].split('\t')[1])
        NumberMissing = int(lines[13].split('\t')[1])
        NumberTotal = int(lines[14].split('\t')[1])
        PercentComplete = 100*(NumberComplete/NumberTotal)
        # Append results to appropriate lists
        Completeness.append(PercentComplete)
        SingleCopies.append(NumberSingleCopy)
        Duplicates.append(NumberDuplicate)
        Fragments.append(NumberFragment)
        Missing.append(NumberMissing)

In [ ]:
## Determine which files have a completeness of greater than 80%

# List comprehenson to return list of indices which have a completeness of 80% or greater
Over80Indices = [ind for ind, comp in enumerate(Completeness) if comp >= 80]
Over80Indices_SCopy = [ind for ind, comp in enumerate(SingleCopies) if comp >= 606]
Under25Indices_Dupl = [ind for ind, comp in enumerate(Duplicates) if comp <= 189]
OverlapIndices = [item for item in Over80Indices if item in Under25Indices_Dupl] ## Check intersection between completeness and duplicate threshold lists


# Use indices determined above to extract filenames of Cryptococcus BUSCO results as new list
Over80Complete = [SelectedBUSCOs_ZipFiles[i] for i in Over80Indices]
Over80SingleCopy = [SelectedBUSCOs_ZipFiles[i] for i in Over80Indices_SCopy]
Under25Duplicate = [SelectedBUSCOs_ZipFiles[i] for i in Under25Indices_Dupl]
Overlap = [SelectedBUSCOs_ZipFiles[i] for i in OverlapIndices]

# Output number of sequences meeting the completeness criterion
print("Number of sequences with greater than 80% completeness: ",len(Over80Complete), " of ", len(SelectedBUSCOs_ZipFiles))
print("Number of sequences with greater than 80% single-copies: ",len(Over80SingleCopy), " of ", len(SelectedBUSCOs_ZipFiles))
print("Number of sequences with less than 25% duplicates: ",len(Under25Duplicate), " of ", len(SelectedBUSCOs_ZipFiles))
print("Number of sequences with greater than 80% completeness with less than 25% duplicates: ",len(Overlap), " of ", len(SelectedBUSCOs_ZipFiles))

In [ ]:
# This weird renaming is just to not require changing the following code which uses this variable name (even if the naming is inaccurate)
Over80Complete = Overlap

In [ ]:
os.makedirs('/path/to/your/analysis_directory/ForGeneTrees/RedoGeneTrees_AlignThenConcatenate/IntermediateFiles/Cryptococcus_Reduced_IF', exist_ok=True)
os.chdir('/path/to/your/analysis_directory/ForGeneTrees/RedoGeneTrees_AlignThenConcatenate/IntermediateFiles/Cryptococcus_Reduced_IF')

# Create a DataFrame
data = {
    'SingleCopies': SingleCopies,
    'Duplicates': Duplicates,
    'Fragments': Fragments,
    'Missing': Missing,
    'Completeness': Completeness,
    'File' : SelectedBUSCOs_ZipFiles
}

df = pd.DataFrame(data)

# Save to a CSV file
output_file = "Cryptococcus_Reduced_BUSCO_Results.csv"
df.to_csv(output_file, index=False)

print(f"DataFrame saved to {output_file}")

In [ ]:
for i in np.arange(len(SingleCopies)):
    print("S:", SingleCopies[i],"; D:", Duplicates[i], "; F:", Fragments[i], "; M:", Missing[i], ";  C:", Completeness[i])

In [ ]:
import io
import tarfile
from pathlib import Path

print("=== Extracting single-copy BUSCO AA files from tar.gz ===")

AAseqFiles = []

for busco_tar in Over80Complete:
    tar_path = Path(file_path) / busco_tar
    single_copy_faas = []

    with tarfile.open(tar_path, "r:gz") as tar:
        # `busco --tar` compresses this folder into a nested tar.gz inside the outer archive
        single_copy_tar_path = 'run_fungi_odb10/busco_sequences/single_copy_busco_sequences.tar.gz'

        try:
            single_copy_tar_member = tar.getmember(single_copy_tar_path)
            single_copy_tar_content = tar.extractfile(single_copy_tar_member).read()

            with tarfile.open(fileobj=io.BytesIO(single_copy_tar_content), mode='r:gz') as inner_tar:
                single_copy_faas = [Path(m).name for m in inner_tar.getnames() if m.endswith(".faa")]

        except KeyError:
            # Fall back to a flat layout, in case BUSCO was run without --tar
            single_copy_faas = [
                Path(m).name for m in tar.getnames()
                if "single_copy_busco_sequences" in m and m.endswith(".faa")
            ]

    if not single_copy_faas:
        print(f"[WARNING] No single-copy BUSCOs found in {busco_tar}")

    AAseqFiles.append(single_copy_faas)

print(f"✓ Loaded BUSCO AA files for {len(AAseqFiles)} assemblies")


In [ ]:
AAseqFilesClean = [[Path(x).name for x in lst] for lst in AAseqFiles]

common_buscos = set(AAseqFilesClean[0])
for file_list in AAseqFilesClean[1:]:
    common_buscos.intersection_update(file_list)

CommonElements = sorted(common_buscos)

print(f"✓ Found {len(CommonElements)} common single-copy BUSCOs across {len(AAseqFilesClean)} assemblies")


In [ ]:
#CommonElements

In [ ]:

import io
import subprocess
import sys
import time
import re
import shutil
import os 
import collections
from pathlib import Path
import numpy as np

## --- SLURM Helper Functions ---
def submit_job(command):
    """Submit an sbatch job and return the job ID."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    print(result.stdout)
    match = re.search(r'Submitted batch job (\d+)', result.stdout)
    if not match:
        raise RuntimeError(f"Failed to submit job: {result.stdout}\n{result.stderr}")
    return match.group(1)

def wait_for_job(job_id, poll_interval=30):
    """Wait until the SLURM job finishes."""
    while True:
        result = subprocess.run(["squeue", "-j", str(job_id)], capture_output=True, text=True)
        if job_id not in result.stdout:
            print(f"Job {job_id} finished.")
            break
        else:
            print(f"Job {job_id} still running... waiting {poll_interval}s")
            time.sleep(poll_interval)
## --- End SLURM Helper Functions ---

# --- Robust FASTA Parsing Function ---
def _robust_fasta_parser(filepath):
    """Parses a FASTA file and returns a dictionary of {header: sequence}."""
    sequences = collections.OrderedDict()
    current_header = None
    current_seq_lines = []
    
    try:
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue 

                if line.startswith('>'):
                    if current_header is not None:
                        sequences[current_header] = "".join(current_seq_lines)
                    
                    # CRITICAL: Extract only the accession ID that will be used as the key.
                    current_header = line.lstrip('>').split('\n')[0].split()[0]
                    current_seq_lines = []
                elif current_header is not None:
                    current_seq_lines.append(line)
        
        # Save the very last sequence
        if current_header is not None:
            sequences[current_header] = "".join(current_seq_lines)

    except Exception as e:
        print(f"Error reading FASTA file {filepath}: {e}")
        return collections.OrderedDict()
        
    return sequences

In [ ]:
print("=== Setting up directories and files for alignment pipeline ===\n")

## --- Define File Paths and Directories ---
family_name = 'Cryptococcus_Reduced' 
experiment_type = 'BUSCO_Trees' 

# Use the variables defined in the preceding cells: CommonElements and Over80Complete
if 'CommonElements' not in locals() or not CommonElements:
    print("[FATAL] CommonElements is empty or undefined. No common BUSCOs shared across all selected assemblies.")
    sys.exit(1)

if 'Over80Complete' not in locals() or not Over80Complete:
    print("[FATAL] Over80Complete is empty or undefined.")
    sys.exit(1)

# Determine stable base directory
BASE_DIR_NAME = f'{experiment_type}_pipeline'
NOTEBOOK_CWD = Path.cwd()
BASE_DIR = NOTEBOOK_CWD / BASE_DIR_NAME

TRIM_DIR = BASE_DIR / f'Trimmed_Alignments_{experiment_type}'
TEMP_FASTA_DIR = BASE_DIR / 'Temp_Gene_Fastas'
SLURM_LOG_DIR = BASE_DIR / 'slurm_logs'

# Ensure directories are created in the new, stable location
BASE_DIR.mkdir(exist_ok=True)
TRIM_DIR.mkdir(exist_ok=True)
TEMP_FASTA_DIR.mkdir(exist_ok=True)
SLURM_LOG_DIR.mkdir(exist_ok=True)
os.chdir(BASE_DIR)

print(f"Working directory: {BASE_DIR}")
print(f"Trim directory: {TRIM_DIR}")
print(f"Temp FASTA directory: {TEMP_FASTA_DIR}")
print(f"SLURM log directory: {SLURM_LOG_DIR}")

# File names for inter-script communication
COMMON_BUSCOS_LIST = 'CommonBUSCOs.txt'
COMPLETE_ASSEMBLIES_LIST = 'Over80CompleteAssemblies.txt'

# Save the list of common BUSCOs (one per line) for the shell script to read
common_buscos_path = BASE_DIR / COMMON_BUSCOS_LIST
with open(common_buscos_path, 'w') as f:
    f.write('\n'.join(CommonElements) + '\n')
print(f"✓ Saved {len(CommonElements)} common BUSCOs to {common_buscos_path}")

# Save the list of complete assembly IDs (folder names) for the shell script to read
complete_assemblies_path = BASE_DIR / COMPLETE_ASSEMBLIES_LIST
with open(complete_assemblies_path, 'w') as f:
    f.write('\n'.join(Over80Complete) + '\n')
print(f"✓ Saved {len(Over80Complete)} assembly files to {complete_assemblies_path}")

total_genes = len(CommonElements)
print(f"\n📊 Pipeline Summary:")
print(f"   Total genes to process in parallel: {total_genes}")
print(f"   Total assemblies used in supermatrix: {len(Over80Complete)}")

In [ ]:
print("=== Creating alignment and trimming script ===\n")

# Path to the packaged BUSCO tar.gz files (see Zip_BUSCO_Results_Cryptococcus.sh)
TAR_FILE_PATH = '/path/to/your/analysis_directory/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/BUSCO_CryptococcusReduced/busco_results'

script_content = """#!/bin/bash
#SBATCH --job-name=Busco_AT_%A_%a
#SBATCH --output=slurm_logs/busco_AT_%a.out
#SBATCH --error=slurm_logs/busco_AT_%a.err
#SBATCH --cpus-per-task=4
#SBATCH --mem=6G
#SBATCH --partition=YOUR_PARTITION   # set this to your cluster's partition/account
#SBATCH --array=1-{}%50

FAMILY=$1
EXP_TYPE=$2
BUSCO_RESULTS_DIR=$3
BASE_DIR=$4

# Path to the extracted BUSCO tar.gz files
TAR_FILE_PATH="{}"

TRIM_DIR="$BASE_DIR/Trimmed_Alignments_$EXP_TYPE"
TEMP_FASTA_DIR="$BASE_DIR/Temp_Gene_Fastas"
mkdir -p "$TRIM_DIR" "$TEMP_FASTA_DIR" slurm_logs

BUSCO_GENE=$(sed -n "${{SLURM_ARRAY_TASK_ID}}p" "$BASE_DIR/CommonBUSCOs.txt")
[[ -z "$BUSCO_GENE" ]] && echo "No gene for task $SLURM_ARRAY_TASK_ID" && exit 1

BUSCO_ID="${{BUSCO_GENE%.faa}}"

echo "--- Processing Gene: $BUSCO_GENE (Task $SLURM_ARRAY_TASK_ID) ---"

gene_fasta="$TEMP_FASTA_DIR/$BUSCO_GENE"
> "$gene_fasta"

extracted=0

# Each outer archive is run_fungi_odb10_ACCESSION.tar.gz (see Zip_BUSCO_Results_Cryptococcus.sh).
# `busco --tar` nests the single-copy sequences inside a second tar.gz within it, so this
# extracts in two stages via pipes rather than looking for a flat internal path.
SINGLE_COPY_TAR_PATH='run_fungi_odb10/busco_sequences/single_copy_busco_sequences.tar.gz'
FAA_MEMBER_PATH="single_copy_busco_sequences/$BUSCO_ID.faa"

while IFS= read -r tar_name; do
    [[ -z "$tar_name" ]] && continue
    TAR_PATH="$TAR_FILE_PATH/$tar_name"
    [[ ! -f "$TAR_PATH" ]] && {{ echo "  WARNING: Tar file not found: $TAR_PATH"; continue; }}

    # Recover the accession from run_fungi_odb10_ACCESSION.tar.gz
    accession="${{tar_name#run_fungi_odb10_}}"
    accession="${{accession%.tar.gz}}"

    seq=$(tar --to-stdout -xzf "$TAR_PATH" "$SINGLE_COPY_TAR_PATH" 2>/dev/null \\
        | tar --to-stdout -xzf - "$FAA_MEMBER_PATH" 2>/dev/null \\
        | grep -v '^>' | tr -d '\\n')

    if [[ -n "$seq" ]]; then
        printf ">%s\\n%s\\n" "$accession" "$seq" >> "$gene_fasta"
        echo "    Extracted $accession"
        ((extracted++))
    else
        echo "    WARNING: $accession is MISSING the gene $BUSCO_GENE"
    fi
done < "$BASE_DIR/Over80CompleteAssemblies.txt"

echo "Total sequences extracted: $extracted"

if (( extracted < 4 )); then
    echo "Skipping $BUSCO_GENE — only $extracted sequences"
    rm -f "$gene_fasta"
    exit 0
fi

mafft --auto --thread 4 --maxiterate 1000 "$gene_fasta" > "$gene_fasta.aln"
trimal -automated1 -in "$gene_fasta.aln" -out "$TRIM_DIR/$BUSCO_GENE.trimmed.faa"

rm -f "$gene_fasta" "$gene_fasta.aln"
echo "Finished $BUSCO_GENE"
"""

# Format the script with total_genes and TAR_FILE_PATH
final_script = script_content.format(total_genes, TAR_FILE_PATH)

# Write it
alignment_script_path = BASE_DIR / "busco_align_trim.sh"
alignment_script_path.write_text(final_script)
alignment_script_path.chmod(0o755)

print(f"SUCCESS: Script created → {alignment_script_path}")
print(f"Tar files will be read from: {TAR_FILE_PATH}")


In [ ]:
print("=== Submitting alignment and trimming jobs ===\n")

# The shell script needs the 'file_path' which defines the root BUSCO directory
busco_results_root = Path(BASE_DIR )

if not busco_results_root.exists():
    print(f"❌ ERROR: BUSCO results directory not found: {busco_results_root}")
    sys.exit(1)

print(f"BUSCO results directory: {busco_results_root}")

# Submit the SLURM array job
align_trim_cmd = (
    f'sbatch '
    f'--job-name={family_name}_{experiment_type}_ATarray '
    f'{alignment_script_path} '
    f'{family_name} {experiment_type} {busco_results_root} {BASE_DIR}'
)

print(f"Submitting {total_genes} alignment/trimming tasks...")
print(f"Command: {align_trim_cmd}")

try:
    align_trim_job = submit_job(align_trim_cmd)
    print(f"✓ Successfully submitted job {align_trim_job}")
    print("Waiting for job completion...")
    wait_for_job(align_trim_job)
    print("✓ Alignment and trimming completed successfully!")
except Exception as e:
    print(f"❌ Failed to submit alignment job: {e}")
    sys.exit(1)

In [ ]:
print("=== Concatenating trimmed alignments into supermatrix ===\n")

Supermatrix_Out_File = f'{family_name}_{experiment_type}_BUSCOsTrimmed_Supermatrix.fa'
Partition_Out_File = f'{family_name}_{experiment_type}_BUSCOsTrimmed_partitions.txt'

# --- Read and Organize Sequences ---
assembly_sequences = {} 
partition_data = []
current_start = 1

# List the trimmed alignment files
trimmed_files = sorted([f for f in TRIM_DIR.iterdir() if f.name.endswith('.trimmed.faa')])
if not trimmed_files:
    print(f"❌ ERROR: No trimmed alignment files found in {TRIM_DIR}")
    print("This usually means the SLURM array job failed.")
    sys.exit(1)

print(f"Found {len(trimmed_files)} trimmed alignment files")

# First, let's check what headers are actually in one of the trimmed files
print("\nChecking headers in first trimmed file...")
sample_file = trimmed_files[0]
with open(sample_file, 'r') as f:
    headers = []
    for i, line in enumerate(f):
        if line.startswith('>'):
            headers.append(line.strip()[1:])  # Remove '>'
        if len(headers) >= 3:  # Check first 3 headers
            break

print("Sample headers from trimmed files:")
for i, header in enumerate(headers[:3]):
    print(f"  {i+1}. >{header}")

# Based on the shell script, headers should be the filename without .tar.gz
# Let's use the actual filenames from Over80Complete (without .tar.gz)
expected_assemblies = [filename.replace('run_fungi_odb10_', '').replace('.tar.gz', '') for filename in Over80Complete]

print(f"\nExpected {len(expected_assemblies)} assemblies")
print("Sample expected assemblies:")
for i, accession in enumerate(expected_assemblies[:3]):
    print(f"  {accession}")

# Pre-populate assembly_sequences with all required accessions initialized to empty strings
for accession_key in expected_assemblies:
    assembly_sequences[accession_key] = ''

print(f"\nProcessing {len(trimmed_files)} gene alignments...")

for filepath in trimmed_files:
    # Gene ID is the filename without the suffix
    gene_id = filepath.name.replace('.trimmed.faa', '')
    if gene_id not in CommonElements:
        print(f"Warning: Trimmed file {filepath.name} is not in CommonElements. Skipping.")
        continue 
        
    gene_length = 0
    
    # Use the robust parser
    gene_sequences = _robust_fasta_parser(filepath)

    if gene_sequences:
        # Get the length of the first sequence read
        if gene_sequences:
            first_key = list(gene_sequences.keys())[0]
            gene_length = len(gene_sequences[first_key])
            print(f"  {gene_id}: {len(gene_sequences)} sequences, length={gene_length}")
    
    if gene_length == 0:
        print(f"WARNING: Skipping alignment file {filepath.name} - Length is zero.")
        continue

    # Update partition data
    partition_data.append(f"WAG, {gene_id} = {current_start}-{current_start + gene_length - 1}")
    current_start += gene_length
    
    # Concatenate sequence for each expected assembly, padding with gaps if missing
    for accession_key in expected_assemblies: 
        # Get the sequence or a gap string if the assembly is missing the gene
        seq = gene_sequences.get(accession_key, '-' * gene_length)
        
        # Ensure sequence length is correct (sanity check)
        if len(seq) != gene_length:
            print(f"WARNING: Length mismatch for {accession_key} in gene {gene_id}. Expected: {gene_length}, Found: {len(seq)}. Padding/Truncating.")
            if len(seq) > gene_length:
                seq = seq[:gene_length]
            else:
                seq = seq + ('-' * (gene_length - len(seq)))
            
        assembly_sequences[accession_key] += seq

print(f"\n✓ Processed all gene alignments")
print(f"Total supermatrix length: {current_start - 1} amino acids")

# Check if we actually got any sequences
non_gap_sequences = {k: v for k, v in assembly_sequences.items() if v.replace('-', '') != ''}
print(f"\nSequences with actual data: {len(non_gap_sequences)}/{len(assembly_sequences)}")

if len(non_gap_sequences) == 0:
    print("\n❌ WARNING: No actual sequences found! All are gaps.")
    print("This suggests the accession keys don't match.")
    print("\nLet's check what headers are actually in the files...")
    
    # Collect all unique headers from all trimmed files
    all_headers = set()
    for filepath in trimmed_files[:5]:  # Check first 5 files
        with open(filepath, 'r') as f:
            for line in f:
                if line.startswith('>'):
                    all_headers.add(line.strip()[1:])
    
    print(f"\nSample headers found in trimmed files:")
    for i, header in enumerate(list(all_headers)[:10]):
        print(f"  {header}")
    
    print(f"\nSample expected headers:")
    for i, header in enumerate(expected_assemblies[:10]):
        print(f"  {header}")
    
    # Try to match by finding common patterns
    print("\nTrying to find matching patterns...")
    for actual_header in list(all_headers)[:3]:
        for expected_header in expected_assemblies[:3]:
            if actual_header in expected_header or expected_header in actual_header:
                print(f"  Possible match: '{actual_header}' <-> '{expected_header}'")

# --- Write the supermatrix FASTA file ---
print(f"\nWriting supermatrix to: {Supermatrix_Out_File}")
with open(Supermatrix_Out_File, 'w') as out_fasta:
    for accession, sequence in assembly_sequences.items():
        out_fasta.write(f'>{accession}\n')
        # Write sequence in chunks of 80 characters for readability
        for i in range(0, len(sequence), 80):
            out_fasta.write(f'{sequence[i:i+80]}\n')

# --- Write the partition file ---
print(f"Writing partition file to: {Partition_Out_File}")
with open(Partition_Out_File, 'w') as out_part:
    out_part.write('\n'.join(partition_data))

print("\n✓ Supermatrix and partition files created!")
print(f"  Supermatrix: {Supermatrix_Out_File}")
print(f"  Partitions: {Partition_Out_File}")

In [ ]:
print("=== Writing supermatrix and partition files ===\n")

# Write Supermatrix
try:
    with open(Supermatrix_Out_File, 'w') as f_supermatrix:
        for assembly_id, sequence in assembly_sequences.items():
            # Write accession ID as the FASTA header
            f_supermatrix.write(f">{assembly_id}\n{sequence}\n")
    print(f"✓ Supermatrix saved to {Supermatrix_Out_File}")
except Exception as e:
    print(f"❌ Failed to write supermatrix: {e}")
    sys.exit(1)

# Write Partition File
try:
    with open(Partition_Out_File, 'w') as f_partition:
        f_partition.write('\n'.join(partition_data) + '\n')
    print(f"✓ Partition file saved to {Partition_Out_File}")
except Exception as e:
    print(f"❌ Failed to write partition file: {e}")
    sys.exit(1)

# Final checks
if os.path.exists(Supermatrix_Out_File) and os.path.getsize(Supermatrix_Out_File) > 0:
    total_length = current_start - 1
    if total_length == 0:
        print("❌ ERROR: Calculated total alignment length is zero!")
        sys.exit(1)
        
    print(f"✅ Concatenation successful!")
    print(f"   Supermatrix: {Supermatrix_Out_File}")
    print(f"   Partition: {Partition_Out_File}")
    print(f"   Total length: {total_length} amino acids")
    print(f"   Number of assemblies: {len(assembly_sequences)}")
    
else:
    print("❌ ERROR: Python concatenation failed. Supermatrix file is missing or empty.")
    sys.exit(1)

# Clean up TEMP_FASTA_DIR 
shutil.rmtree(TEMP_FASTA_DIR, ignore_errors=True)
print(f"✓ Cleaned up temporary directory: {TEMP_FASTA_DIR}")

# Save the supermatrix filename for the IQ-Tree step
FinalAlignmentFile = Supermatrix_Out_File
print(f"✓ Final alignment file ready for IQ-Tree: {FinalAlignmentFile}")

In [ ]:
from pathlib import Path

# --- Set your original partition file ---
Partition_Out_File = "Cryptococcus_Reduced_BUSCO_Trees_BUSCOsTrimmed_partitions.txt"
corrected_partition_file = Partition_Out_File.replace('.txt', '_iqtree_spp_ready.txt')

# --- Read original partition file ---
with open(Partition_Out_File, 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

# --- Filter lines that contain '=' and strip any unwanted whitespace ---
clean_lines = []
for line in lines:
    if '=' in line:
        # Remove leading/trailing whitespace around commas and '='
        parts = line.split('=')
        left, right = parts[0].strip(), parts[1].strip()
        
        # Keep model and gene name intact (everything before '=')
        clean_lines.append(f"{left} = {right}")

# --- Save corrected file compatible with -spp ---
with open(corrected_partition_file, 'w') as f:
    f.write('\n'.join(clean_lines))

print(f"✓ Partition file rebuilt and ready for -spp: {corrected_partition_file}")

In [ ]:
print("=== Running IQ-Tree with absolute paths ===\n")

# Cancel any previous failed jobs
subprocess.run(f"scancel -n {family_name}_{experiment_type}_iqtree", shell=True, capture_output=True)

# Use absolute paths to ensure files are found
tree_cmd = (
    f'sbatch --wrap="cd {BASE_DIR} && iqtree2 '
    f'-s {FinalAlignmentFile} '
    f'-spp {corrected_partition_file} '
    f'-T AUTO -m MFP+MERGE -bb 1000 -alrt 1000 --redo" '
    f'--job-name={family_name}_{experiment_type}_iqtree '
    f'--output={family_name}_{experiment_type}_iqtreeOutput_%j.out '
    f'--error={family_name}_{experiment_type}_iqtreeError_%j.err '
    '--cpus-per-task=32 --mem=32G --partition=YOUR_PARTITION'
)

print(f"Command: {tree_cmd}")

try:
    tree_job = submit_job(tree_cmd)
    print(f"✓ Successfully submitted IQ-Tree job {tree_job}")
    print("Waiting for job completion...")
    wait_for_job(tree_job)
    print("✅ IQ-Tree job completed successfully!")
    print("All jobs completed successfully.")
except Exception as e:
    print(f"❌ Failed: {e}")

## Plot tree, and examine renamed labels

In [ ]:
## Display maximum likelihood tree determined using IQtree2


from Bio import Phylo




os.chdir('/path/to/your/analysis_directory/ForGeneTrees/RedoGeneTrees_AlignThenConcatenate/IntermediateFiles/Cryptococcus_Reduced_IF/BUSCO_Trees_pipeline')


# Load the IQ-TREE output file (convert to Newick format if necessary)
tree = Phylo.read("Cryptococcus_Reduced_BUSCO_Trees_BUSCOsTrimmed_partitions_iqtree_spp_ready.txt.treefile", "newick")

# Plot the tree
fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(1, 1, 1)
Phylo.draw(tree, axes=ax)
plt.show()

In [ ]:
Over80Accessions = [filename.replace('run_fungi_odb10_', '').replace('.tar.gz', '') for filename in Over80Complete]

Over80Accessions

In [ ]:
Cryptococcus_Reduced_AccessionToSpecies = {}

# Build a dict: accession → species (accession)
acc_to_species = {
    acc: f"{Cryptococcus_df.loc[Cryptococcus_df['Assembly Accession'] == acc, 'Organism Name'].values[0]} ({acc})"
    for acc in Over80Accessions
    if acc in Cryptococcus_df['Assembly Accession'].values
}

# Map full filenames (without .tar.gz) → species (accession)
for fname in Over80Complete:
    # remove the .tar.gz suffix
    clean_fname = fname.replace(".tar.gz", "")
    
    # extract accession prefix
    acc = "_".join(fname.split("_")[0:2])
    
    if acc in acc_to_species:
        Cryptococcus_Reduced_AccessionToSpecies[clean_fname] = acc_to_species[acc]
    else:
        Cryptococcus_Reduced_AccessionToSpecies[clean_fname] = "UNKNOWN"

Cryptococcus_Reduced_AccessionToSpecies

In [ ]:
## Save the dictionary of labels for reuse later

import pickle

# Save dictionary to a file
with open('Cryptococcus_ReducedDictionary.pkl', 'wb') as file:
    pickle.dump(Cryptococcus_Reduced_AccessionToSpecies, file)

In [ ]:
# Load the IQ-TREE output file (Newick format)
tree = Phylo.read("Cryptococcus_Reduced_BUSCO_Trees_BUSCOsTrimmed_partitions_iqtree_spp_ready.txt.treefile", "newick")

# Update the leaf labels in the tree
for leaf in tree.get_terminals():
    if leaf.name in Cryptococcus_Reduced_AccessionToSpecies:
        leaf.name = Cryptococcus_Reduced_AccessionToSpecies[leaf.name]

# Plot the updated tree
fig = plt.figure(figsize=(10, 15))
ax = fig.add_subplot(1, 1, 1)
Phylo.draw(tree, axes=ax)
plt.show()